In [2]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../warehouse/payments_warehouse.db")

# Full fact table joined with all dimensions - one flat file, easiest for Power BI to model
export_query = """
SELECT
    f.transaction_id, d.full_date, d.year, d.month, d.month_name, d.day_name, d.is_weekend, d.quarter,
    f.hour_of_day, g.sender_state, mc.merchant_category, tt.transaction_type,
    dv.device_type, n.network_type, ag.age_group,
    f.amount_inr, f.transaction_status, f.fraud_flag
FROM fact_transactions f
JOIN dim_date d ON f.date_key = d.date_key
JOIN dim_geography g ON f.sender_state_key = g.sender_state_key
JOIN dim_merchant_category mc ON f.merchant_category_key = mc.merchant_category_key
JOIN dim_transaction_type tt ON f.transaction_type_key = tt.transaction_type_key
JOIN dim_device dv ON f.device_type_key = dv.device_type_key
JOIN dim_network n ON f.network_type_key = n.network_type_key
JOIN dim_age_group ag ON f.age_group_key = ag.age_group_key
"""

dashboard_export = pd.read_sql(export_query, conn)
dashboard_export.to_csv("../dashboards/data/transactions_flat.csv", index=False)

# Also bring in your Phase 7 clusters and Phase 8 forecast comparison - Power BI can join these separately
segment_clusters = pd.read_csv("../data/processed/segment_clusters_labeled.csv")
segment_clusters.to_csv("../dashboards/data/segment_clusters.csv", index=False)

forecast_comparison = pd.read_csv("../reports/forecast_model_comparison.csv")
forecast_comparison.to_csv("../dashboards/data/forecast_comparison.csv", index=False)

print(dashboard_export.shape)

(250000, 18)


In [3]:
import pandas as pd

check = pd.read_csv("../dashboards/data/transactions_flat.csv")
print(check["fraud_flag"].dtype)
print(check["fraud_flag"].unique())
print(check["fraud_flag"].sum())

int64
[0 1]
480
